# MirrorTopology Step 1 — Phase D-1：第 1 波 30 配置の production 共分散生成と intake（v0.3；実行前監査 RD1-A/B/C・RD12-A/B/C 反映）
A11 の登録生成手順（凍結 notebook v1.4.1 cell 4 を逐語再現；pinned CMBtopology `0cc65e34`・l_max=4・tag 別 scratch・atomic cache・env fingerprint 付き manifest）で，source-bound registry／Phase C 登録 grid manifest の 30 配置（x₀_CT = −r_obs；E1 は等質）の共分散を生成し，配置ごとに production intake（幾何 binding・run profile・環境・array SHA 再計算・reality／PSD・principal sqrt gate）を記録する。A11 official の 1 entry を同環境で再生成して照合（rel ≤ 1e-10）。登録数値環境（rules §12.1）は生成 process 内の hard gate；Phase C member・保存 grid・凍結 loader・A11 生成 cell・比較先の信頼済み同一性を生成前に検査；Git 照会の終了コード検査；最初の intake 失敗（False／例外とも）で partial registry を保存して停止；保存 grid の意味検証は A11 再生成の前；OUT は fresh-only。所要時間の目安：3〜4 h（Colab CPU；A11 実測 ~400 s／共分散）。label なし・bank 生成なし。

In [ ]:
# --- 0. OUTER LAUNCHER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the verification target>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
LAUNCHER_ID = 'MirrorTopology_Step1_D1_covgen_v0.3'


In [ ]:
# --- 1. fresh scratch checkout at C; inventory bound; d/ pins+script bound; Phase C packet present at the same commit
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256), 'launcher lock not filled'
RUN=f'/content/d1_runs/{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}'; SCRATCH=f'{RUN}/scratch'; OUT=f'{RUN}/out'; os.makedirs(OUT)
subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
head=subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip(); assert head==REPO_COMMIT, head
assert subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()=='', 'scratch tree not clean'
MT=SCRATCH; PHASEB=f'{MT}/engine/phaseB'; PHASEC=f'{MT}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; inv_sha=sha(INV); assert inv_sha==EXPECTED_INVENTORY_SHA256, inv_sha
inv=json.load(open(INV)); PINS=f'{PHASEB}/d/d1_pins.json'; SCRIPT=f'{PHASEB}/d/d1_covgen.py'
assert sha(PINS)==inv['d_sha256']['d/d1_pins.json'] and sha(SCRIPT)==inv['d_sha256']['d/d1_covgen.py']
pins=json.load(open(PINS)); assert pins['schema']=='d1_pins_v1' and pins['engine_version']==inv['engine_version']
assert sha(f'{PHASEC}/PACKET_INVENTORY.json')==pins['first_wave'].get('phaseC_inventory_sha256', sha(f'{PHASEC}/PACKET_INVENTORY.json')), 'Phase C packet differs'
lock=dict(launcher_id=LAUNCHER_ID, repo_url=REPO_URL, commit=head, inventory_sha256=inv_sha, pins_sha256=sha(PINS), engine_version=inv['engine_version'], run_dir=RUN); json.dump(lock, open(f'{OUT}/launcher_lock.json','w'), indent=1); print(lock)


In [ ]:
# --- 2. pinned CMBtopology: fresh isolated checkout; requirements install (A11 procedure); dependency presence checked by the script
CT=f'{RUN}/CMBtopology_pinned'
subprocess.run(['git','clone','-q',pins['cmbtopology']['url']+'.git',CT],check=True); subprocess.run(['git','-C',CT,'checkout','-q',pins['cmbtopology']['commit']],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','healpy==1.20.0'],check=True); subprocess.run([sys.executable,'-m','pip','install','-q','-r',f'{CT}/requirements.txt'],check=True)
ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl'],check=True)   # registered environment (pins); the script re-verifies live versions and pools as a HARD gate
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
print('CMBtopology', subprocess.check_output(['git','-C',CT,'rev-parse','HEAD']).decode().strip())


In [ ]:
# --- 3. D-1 generation + intake (long: ~30 x 400 s + cross-check)
DO=f'{OUT}/d1'
rc=subprocess.run([sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--ct',CT,'--out',DO,'--profile','production_official'],capture_output=True,text=True)
open(f'{OUT}/launcher_script_stdout.txt','w').write(rc.stdout); open(f'{OUT}/launcher_script_stderr.txt','w').write(rc.stderr); print(rc.stdout[-3000:])
rm=json.load(open(f'{DO}/d1_run_manifest.json')); script_ok=bool(rc.returncode==0 and rm.get('D1_PASS') is True); print('script rc', rc.returncode, 'D1_PASS', rm.get('D1_PASS'), rm.get('failures'))


In [ ]:
# --- 4. final record + zip (cov_cache is small: 30 x ~7 KB)
def inventory(root, exclude=()):
    out={}
    for d,_,fs in os.walk(root):
        for f in fs:
            p=os.path.join(d,f); rel=os.path.relpath(p, root)
            if rel in exclude: continue
            out[rel]=dict(sha256=sha(p), bytes=os.path.getsize(p))
    return out
final=dict(launcher=lock, D1_PASS=bool(script_ok), stages=dict(checkout=True, script_returncode=rc.returncode, script_pass=rm.get('D1_PASS'), script_gates=rm.get('gates'), script_failures=rm.get('failures'), env_lock=rm.get('env_lock'), a11_cross_check=rm.get('a11_cross_check'), n_generated=rm.get('n_generated'), n_intake_pass=rm.get('n_intake_pass')))
final['output_inventory']=inventory(OUT, exclude=('d1_final_record.json','d1_return_list.json'))
json.dump(final, open(f'{OUT}/d1_final_record.json','w'), indent=1); json.dump(dict(final_record_sha256=sha(f'{OUT}/d1_final_record.json'), files=final['output_inventory']), open(f'{OUT}/d1_return_list.json','w'), indent=1)
print(json.dumps({k:final[k] for k in ('D1_PASS',)}, indent=1), 'run dir:', RUN)
import shutil
from google.colab import files
p = shutil.make_archive(f'/content/d1_out_{REPO_COMMIT[:12]}', 'zip', root_dir=OUT); print(p, os.path.getsize(p)); files.download(p)
